# Potencia y estudios piloto

Complemento de las páginas del curso *Tamaño muestral y potencia* y *Cómo: planificar un tamaño muestral*.

Todo lo que ves aquí está simulado a partir de un efecto verdadero conocido, así que puedes ver:

1. Cómo crece la **potencia** con el tamaño muestral (y que una simulación coincide con la fórmula de los manuales).
2. Por qué el tamaño del efecto de un **piloto pequeño** es un mal dato de partida para un cálculo de potencia: los pilotos que parecen convincentes son los que exageran.
3. Qué ocurre con un estudio **planificado a partir del tamaño del efecto de un piloto**.

Cambia los parámetros de la siguiente celda y usa *Run All*. Necesita `numpy`, `scipy`, `statsmodels`, `matplotlib`.

In [ ]:
# ---- Parameters: change these and re-run ----
TRUE_D = 0.3          # true standardized effect (Cohen's d = difference / SD); 0.3 is a modest effect
PILOT_N = 10          # units per group in each pilot study
ALPHA = 0.05          # significance level
TARGET_POWER = 0.8    # power you aim for when planning the full study
N_GRID = [10, 20, 40, 80, 120, 160, 240]   # per-group sample sizes for experiment 1
N_SIMS = 4000         # simulated studies per setting
SEED = 0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import TTestIndPower

rng = np.random.default_rng(SEED)
power_calc = TTestIndPower()


def simulate_studies(n, d, n_sims=N_SIMS):
    """Simulate n_sims two-group studies (n per group); return observed d and p-value for each."""
    control = rng.normal(0, 1, (n_sims, n))
    treated = rng.normal(d, 1, (n_sims, n))
    p = stats.ttest_ind(treated, control, axis=1).pvalue
    pooled_sd = np.sqrt((treated.var(axis=1, ddof=1) + control.var(axis=1, ddof=1)) / 2)
    d_hat = (treated.mean(axis=1) - control.mean(axis=1)) / pooled_sd
    return d_hat, p

## 1. Curva de potencia: simulación vs. fórmula

La potencia es la proporción de estudios simulados que alcanzan p < alfa. Los puntos son simulados; la línea es la respuesta de forma cerrada de `statsmodels`. Deberían coincidir, y por eso la simulación es una alternativa fiable para diseños que no tienen fórmula.

In [ ]:
simulated = [np.mean(simulate_studies(n, TRUE_D)[1] < ALPHA) for n in N_GRID]
formula = power_calc.power(effect_size=TRUE_D, nobs1=np.array(N_GRID), alpha=ALPHA)
needed = power_calc.solve_power(effect_size=TRUE_D, alpha=ALPHA, power=TARGET_POWER)
print(f"Per-group n for {TARGET_POWER:.0%} power at d = {TRUE_D}: {int(np.ceil(needed))}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(N_GRID, formula, label="formula")
ax.plot(N_GRID, simulated, "o", label="simulation")
ax.axhline(TARGET_POWER, color="gray", linestyle=":")
ax.set_xlabel("units per group")
ax.set_ylabel("power")
ax.legend()
plt.show()

## 2. Qué te dice un piloto pequeño sobre el efecto

Ejecutamos muchos pilotos de `PILOT_N` unidades por grupo, todos a partir del mismo efecto verdadero. Fíjate en (a) cuánto varía el tamaño del efecto observado y (b) qué ocurre si solo miras los pilotos que resultaron estadísticamente significativos, que son los que tienen más probabilidades de publicarse y de recibir seguimiento.

In [ ]:
d_hat, p = simulate_studies(PILOT_N, TRUE_D)
significant = p < ALPHA

print(f"True d: {TRUE_D}")
print(f"Pilots reaching p < {ALPHA} (= power at n = {PILOT_N}): {significant.mean():.0%}")
print(f"Observed d, all pilots:          mean {d_hat.mean():.2f}, middle 90% from {np.percentile(d_hat, 5):.2f} to {np.percentile(d_hat, 95):.2f}")
print(f"Observed d, significant pilots:  mean {np.abs(d_hat[significant]).mean():.2f}  "
      f"(exaggeration factor {np.abs(d_hat[significant]).mean() / TRUE_D:.1f}x)")
print(f"Significant pilots with the WRONG sign: {(d_hat[significant] < 0).mean():.1%}")

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(d_hat.min(), d_hat.max(), 50)
ax.hist(d_hat[~significant], bins=bins, alpha=0.6, label="not significant")
ax.hist(d_hat[significant], bins=bins, alpha=0.6, label="significant")
ax.axvline(TRUE_D, color="green", linestyle="--", label="true d")
ax.set_xlabel("observed d in the pilot")
ax.set_ylabel("number of pilots")
ax.legend()
plt.show()

## 3. Planificar el estudio completo a partir del tamaño del efecto del piloto

Supongamos que el efecto observado en cada piloto se toma al pie de la letra para dimensionar el estudio completo (n aproximado por grupo con la aproximación normal). Después preguntamos qué potencia tiene realmente ese estudio completo, dado el efecto *verdadero*. Lo repetimos con todos los pilotos y solo con los significativos.

In [ ]:
# Rough per-group n from the pilot's d (normal approximation), capped at 1000 to keep the numbers sensible.
z = stats.norm.ppf(1 - ALPHA / 2) + stats.norm.ppf(TARGET_POWER)
planned_n = np.clip(np.ceil(2 * (z / np.maximum(np.abs(d_hat), 1e-6)) ** 2), 2, 1000)
achieved_power = power_calc.power(effect_size=TRUE_D, nobs1=planned_n, alpha=ALPHA)

print(f"Per-group n that gives {TARGET_POWER:.0%} power at the TRUE d: {int(np.ceil(needed))}")
for label, mask in [("all pilots", np.ones_like(significant)), ("only significant pilots", significant)]:
    print(f"\n{label}:")
    print(f"  planned n per group: median {np.median(planned_n[mask]):.0f}, middle 90% from "
          f"{np.percentile(planned_n[mask], 5):.0f} to {np.percentile(planned_n[mask], 95):.0f}")
    print(f"  true power of the planned study: median {np.median(achieved_power[mask]):.0%}")
    print(f"  planned studies with true power below {TARGET_POWER:.0%}: {(achieved_power[mask] < TARGET_POWER).mean():.0%}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(achieved_power[significant], bins=20, alpha=0.6, label="planned from significant pilots")
ax.hist(achieved_power, bins=20, alpha=0.6, label="planned from all pilots")
ax.axvline(TARGET_POWER, color="gray", linestyle=":")
ax.set_xlabel("true power of the planned study")
ax.set_ylabel("number of pilots")
ax.legend()
plt.show()

## Cosas para probar

- Aumenta `PILOT_N` a 100: la exageración entre los pilotos significativos baja de aproximadamente 4x a 1.4x y los tamaños muestrales planificados se vuelven mucho menos erráticos. El problema es propio de los pilotos pequeños y ruidosos.
- Pon `TRUE_D = 0.8` (un efecto grande): un piloto de 10 por grupo lo detecta aproximadamente el 40 % de las veces, y la exageración es menor (alrededor de 1.7x). La inflación importa sobre todo en los efectos modestos.
- Pon `TRUE_D = 0.1`: solo alrededor del 6 % de los pilotos son significativos, y aproximadamente una cuarta parte de ellos tiene el signo equivocado.
- La alternativa a fiarse de un piloto: elegir el efecto más pequeño que importaría en la práctica y planificar para ese (la tabla de sensibilidad de la página práctica). Prueba `TRUE_D = 0.3` pero planifica directamente con `d = 0.3`, y compara con los tamaños muestrales planificados a partir de los pilotos anteriores.

Ten en cuenta que un piloto sigue siendo valioso para comprobar la logística y estimar la desviación típica y la tasa de abandono. Lo que no debe tomarse al pie de la letra es el *tamaño del efecto* de un piloto pequeño.